# ☁️ CloudVault
## Smart Object Storage Manager

### Tugas Besar Mata Kuliah Cloud Computing

---

**Nama** : Muhammad Yusuf Anwar

**NIM** : (32602500068)

**Kelas** : (RPL LP MAARIF)

---

### Deskripsi

CloudVault adalah aplikasi sederhana untuk mengelola layanan Object Storage (Amazon S3) menggunakan MiniStack sebagai emulator AWS. Aplikasi ini menyediakan fitur pembuatan bucket, manajemen file, pencarian file, serta dashboard statistik melalui antarmuka interaktif berbasis Jupyter Notebook.

---

In [51]:
# ======================================================
# IMPORT LIBRARY
# ======================================================

import boto3
from botocore.exceptions import ClientError
from datetime import datetime
import os

print("✅ Semua library berhasil diimport")

✅ Semua library berhasil diimport


In [52]:
# ======================================================
# KONEKSI KE MINISTACK
# ======================================================

ENDPOINT = "http://127.0.0.1:4566"
REGION = "us-east-1"

ACCESS_KEY = "test"
SECRET_KEY = "test"

s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    region_name=REGION,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY
)

print("=" * 50)
print("☁️ CLOUDVAULT")
print("Berhasil terhubung ke MiniStack")
print("=" * 50)

☁️ CLOUDVAULT
Berhasil terhubung ke MiniStack


In [53]:
# ======================================================
# CEK KONEKSI
# ======================================================

response = s3.list_buckets()

print("Status Koneksi : BERHASIL")
print(f"Jumlah Bucket : {len(response['Buckets'])}")

Status Koneksi : BERHASIL
Jumlah Bucket : 1


In [54]:
bucket_name = "cloudvault-storage"

try:
    s3.create_bucket(Bucket=bucket_name)
    print("Bucket berhasil dibuat")
except Exception as e:
    print(e)

Bucket berhasil dibuat


In [55]:
response = s3.list_buckets()

for bucket in response["Buckets"]:
    print(bucket["Name"])

cloudvault-storage


In [56]:
s3.upload_file(
    Filename="contoh.txt",
    Bucket=bucket_name,
    Key="contoh.txt"
)

print("Upload berhasil")

Upload berhasil


In [57]:
objects = s3.list_objects_v2(Bucket=bucket_name)

for obj in objects.get("Contents", []):
    print(obj["Key"])

contoh.txt
contoh2.txt


In [58]:
s3.download_file(
    bucket_name,
    "contoh.txt",
    "hasil_download.txt"
)

print("Download selesai")

Download selesai


In [59]:
s3.delete_object(
    Bucket=bucket_name,
    Key="contoh.txt"
)

print("File dihapus")

File dihapus


In [60]:
response = s3.list_objects_v2(Bucket=bucket_name)

print(response.get("Contents"))

[{'Key': 'contoh2.txt', 'LastModified': datetime.datetime(2026, 7, 19, 9, 6, 52, 553000, tzinfo=tzutc()), 'ETag': '"c67b5dcba29ef74a6168c95a82c91db5"', 'Size': 39, 'StorageClass': 'STANDARD'}]


## Upload File

In [61]:
def upload_file(local_file, object_name=None):
    """
    Upload file ke bucket S3
    """

    if object_name is None:
        object_name = local_file

    try:
        s3.upload_file(
            local_file,
            bucket_name,
            object_name
        )
        print(f"✅ {object_name} berhasil diupload")

    except Exception as e:
        print("❌ Upload gagal")
        print(e)

In [62]:
upload_file("contoh2.txt")

✅ contoh2.txt berhasil diupload


In [63]:
response = s3.list_objects_v2(Bucket=bucket_name)

for obj in response.get("Contents", []):
    print(obj["Key"])

contoh2.txt


## List Files

In [64]:
def list_files():
    """
    Menampilkan semua file dalam bucket
    """

    response = s3.list_objects_v2(Bucket=bucket_name)

    if "Contents" not in response:
        print("📂 Bucket masih kosong.")
        return

    print("=" * 50)
    print("Daftar File dalam Bucket")
    print("=" * 50)

    for i, obj in enumerate(response["Contents"], start=1):
        print(f"{i}. {obj['Key']} ({obj['Size']} byte)")

In [65]:
list_files()

Daftar File dalam Bucket
1. contoh2.txt (39 byte)


## 📥 Download File

In [66]:
def download_file(object_name, save_as=None):
    """
    Mengunduh file dari bucket ke komputer
    """

    if save_as is None:
        save_as = object_name

    try:
        s3.download_file(
            bucket_name,
            object_name,
            save_as
        )

        print(f"✅ File '{object_name}' berhasil didownload")
        print(f"📁 Disimpan sebagai : {save_as}")

    except Exception as e:
        print("❌ Download gagal")
        print(e)

In [67]:
s3.download_file(
    bucket_name,
    "contoh2.txt",
    "hasil_download.txt"
)

print("File berhasil didownload")

File berhasil didownload


In [68]:
list_files()

Daftar File dalam Bucket
1. contoh2.txt (39 byte)


## 🗑 Delete File

In [69]:
def delete_file(object_name):
    """
    Menghapus file dari bucket
    """

    try:
        s3.delete_object(
            Bucket=bucket_name,
            Key=object_name
        )

        print(f"🗑 File '{object_name}' berhasil dihapus")

    except Exception as e:
        print("❌ Gagal menghapus file")
        print(e)

In [70]:
delete_file("contoh2.txt")

🗑 File 'contoh2.txt' berhasil dihapus


In [71]:
list_files()

📂 Bucket masih kosong.


## 🔍 Search File

In [72]:
def search_file(keyword):
    """
    Mencari file berdasarkan nama
    """

    response = s3.list_objects_v2(Bucket=bucket_name)

    if "Contents" not in response:
        print("📂 Bucket kosong")
        return

    hasil = []

    for obj in response["Contents"]:
        if keyword.lower() in obj["Key"].lower():
            hasil.append(obj)

    if len(hasil) == 0:
        print("❌ File tidak ditemukan")
    else:
        print("=" * 45)
        print("🔍 Hasil Pencarian")
        print("=" * 45)

        for i, obj in enumerate(hasil, start=1):
            print(f"{i}. {obj['Key']} ({obj['Size']} byte)")

In [73]:
search_file("abc")

📂 Bucket kosong


In [74]:
upload_file("contoh2.txt")

✅ contoh2.txt berhasil diupload


In [75]:
upload_file("contoh.txt")

✅ contoh.txt berhasil diupload


In [76]:
search_file("contoh")

🔍 Hasil Pencarian
1. contoh.txt (163 byte)
2. contoh2.txt (39 byte)


In [77]:
search_file("acb")

❌ File tidak ditemukan


## Storage Statistics

Menampilkan informasi statistik penyimpanan pada bucket, seperti jumlah file dan total ukuran file yang tersimpan.

In [78]:
def storage_statistics():
    response = s3.list_objects_v2(Bucket=bucket_name)

    if "Contents" not in response:
        print("="*45)
        print("📊 Storage Statistics")
        print("="*45)
        print("Jumlah File : 0")
        print("Total Size  : 0 Byte")
        return

    total_file = len(response["Contents"])
    total_size = sum(obj["Size"] for obj in response["Contents"])

    print("="*45)
    print("📊 Storage Statistics")
    print("="*45)
    print(f"Jumlah File : {total_file}")
    print(f"Total Size  : {total_size} Byte")
    print(f"            : {total_size/1024:.2f} KB")
    print(f"            : {total_size/1024/1024:.2f} MB")

In [79]:
storage_statistics()

📊 Storage Statistics
Jumlah File : 2
Total Size  : 202 Byte
            : 0.20 KB
            : 0.00 MB


## Bucket Information

Menampilkan informasi dasar mengenai bucket yang digunakan pada aplikasi CloudVault.

In [80]:
def bucket_information():
    response = s3.list_objects_v2(Bucket=bucket_name)

    print("="*45)
    print("🪣 Bucket Information")
    print("="*45)

    print("Bucket Name :", bucket_name)

    if "Contents" not in response:
        print("Jumlah File : 0")
    else:
        print("Jumlah File :", len(response["Contents"]))

In [81]:
bucket_information()

🪣 Bucket Information
Bucket Name : cloudvault-storage
Jumlah File : 2


## Exit Program

Fungsi sederhana untuk menutup atau mengakhiri penggunaan aplikasi CloudVault.

In [82]:
def exit_program():
    print("="*45)
    print("Terima kasih telah menggunakan CloudVault")
    print("Program selesai.")
    print("="*45)

In [83]:
exit_program()

Terima kasih telah menggunakan CloudVault
Program selesai.


# Kesimpulan

CloudVault berhasil diimplementasikan menggunakan Python, Boto3, dan MiniStack sebagai emulator Amazon S3.

Fitur yang berhasil dibuat:

- Membuat bucket
- Menampilkan daftar bucket
- Upload file
- Menampilkan daftar file
- Download file
- Menghapus file
- Mencari file
- Menampilkan statistik storage
- Menampilkan informasi bucket
- Exit aplikasi

Seluruh fitur telah berhasil diuji menggunakan MiniStack tanpa menggunakan akun AWS asli.